# Notebook 09 — Evaluation and Publication Figures

**Purpose:** Generate all publication-quality figures and final metric tables.

**Outputs:** All figures in `figures/` at 300 DPI, suitable for journal submission.

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else '.')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from pathlib import Path

from src.utils import (
    MODELS_DIR, FIGURES_DIR, PROCESSED_DIR, PREDICTIONS_DIR,
    TERRAIN_CLASSES, CLASS_COLORS, NUM_CLASSES, get_logger,
)
from src.metrics import compute_all_metrics, print_metrics_table, compute_confusion_matrix

log = get_logger('09_evaluation')

# Publication style
plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'figure.dpi': 100,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

## 1. Comprehensive Model Comparison Table

In [ ]:
# Collect all model metrics
metric_files = sorted(MODELS_DIR.glob('*_metrics.json'))

all_results = []
for mf in metric_files:
    with open(mf) as f:
        m = json.load(f)
    
    name = m.get('run_name', m.get('model', mf.stem.replace('_metrics', '')))
    
    row = {'Model': name}
    
    # Extract metrics depending on format
    if 'test' in m and isinstance(m['test'], dict):
        test = m['test']
        row['Accuracy'] = test.get('overall_accuracy', None)
        row['Mean IoU'] = test.get('mean_iou', None)
        for cls_name in TERRAIN_CLASSES.values():
            row[f'IoU_{cls_name}'] = test.get('per_class_iou', {}).get(cls_name, None)
            row[f'F1_{cls_name}'] = test.get('per_class_f1', {}).get(cls_name, None)
    elif 'test_iou' in m:
        row['Mean IoU'] = m['test_iou']
        row['Best Val IoU'] = m.get('best_val_iou', None)
    
    all_results.append(row)

results_df = pd.DataFrame(all_results)
if 'Mean IoU' in results_df.columns:
    results_df = results_df.sort_values('Mean IoU', ascending=False)

print("Model Comparison (sorted by Mean IoU):")
print(results_df.to_string(index=False, float_format='%.4f'))

# Save
results_df.to_csv(FIGURES_DIR / 'model_comparison.csv', index=False)

## 2. Confusion Matrices

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

class_names = [TERRAIN_CLASSES[i] for i in range(NUM_CLASSES)]

# Find RF and best DL predictions
rf_preds_path = PREDICTIONS_DIR / 'rf_baseline_test_predictions.npy'
dl_pred_files = sorted(PREDICTIONS_DIR.glob('*_test_predictions.npy'))
dl_pred_files = [f for f in dl_pred_files if 'rf_baseline' not in f.name]

# Load test labels
from src.utils import SPLITS_DIR
with open(SPLITS_DIR / 'split_v1.json') as f:
    split_map = json.load(f)
test_ids = sorted([k for k, v in split_map.items() if v == 'test'])

LABEL_TILES_DIR = PROCESSED_DIR / 'label_tiles'
test_labels = []
for tid in test_ids:
    lbl = np.load(LABEL_TILES_DIR / f"{tid}.npy")
    valid = lbl[lbl != 255]
    if len(valid) > 0:
        vals, counts = np.unique(valid, return_counts=True)
        test_labels.append(int(vals[np.argmax(counts)]))

test_labels = np.array(test_labels)

n_plots = 1 + len(dl_pred_files)
fig, axes = plt.subplots(1, min(n_plots, 3), figsize=(6 * min(n_plots, 3), 5))
if n_plots == 1:
    axes = [axes]

plot_idx = 0

# RF confusion matrix
if rf_preds_path.exists():
    rf_preds = np.load(rf_preds_path)
    if len(rf_preds) == len(test_labels):
        ConfusionMatrixDisplay.from_predictions(
            test_labels, rf_preds,
            display_labels=class_names,
            normalize='true',
            ax=axes[plot_idx],
            cmap='Blues',
            values_format='.2f',
        )
        axes[plot_idx].set_title('RF Baseline')
        axes[plot_idx].tick_params(axis='x', rotation=45)
        plot_idx += 1

# DL confusion matrices
for pred_file in dl_pred_files[:2]:  # Max 2 DL models
    if plot_idx >= len(axes):
        break
    dl_preds = np.load(pred_file)
    name = pred_file.stem.replace('_test_predictions', '')
    # DL preds are pixel-level; need majority vote per tile for comparison
    # or compare at pixel level if shapes match
    if len(dl_preds.shape) > 1:
        # Pixel-level predictions — take majority vote
        tile_preds = []
        for pred_tile in dl_preds:
            vals, counts = np.unique(pred_tile[pred_tile != 255], return_counts=True)
            tile_preds.append(int(vals[np.argmax(counts)]) if len(vals) > 0 else 0)
        tile_preds = np.array(tile_preds)
    else:
        tile_preds = dl_preds
    
    if len(tile_preds) == len(test_labels):
        ConfusionMatrixDisplay.from_predictions(
            test_labels, tile_preds,
            display_labels=class_names,
            normalize='true',
            ax=axes[plot_idx],
            cmap='Oranges',
            values_format='.2f',
        )
        axes[plot_idx].set_title(name)
        axes[plot_idx].tick_params(axis='x', rotation=45)
        plot_idx += 1

plt.suptitle('Normalised Confusion Matrices (Test Set)', fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'confusion_matrices.png', dpi=300)
plt.show()

## 3. Qualitative Comparison Grid

In [ ]:
# SAR → RF prediction → DL prediction → ground truth
# For representative tiles of each terrain type

SAR_TILES_DIR = PROCESSED_DIR / 'sar_tiles'
tile_meta = pd.read_csv(PROCESSED_DIR / 'tile_metadata.csv')

class_cmap = ListedColormap([CLASS_COLORS[i] for i in range(NUM_CLASSES)])

# Select 1 representative tile per class from test set
test_meta = tile_meta[tile_meta['tile_id'].isin(test_ids)]
representative_tiles = []
for cls in range(NUM_CLASSES):
    col = f'class_{cls}_frac'
    if col in test_meta.columns:
        best = test_meta.nlargest(1, col)
        if len(best) > 0:
            representative_tiles.append(best.iloc[0]['tile_id'])

n_tiles = len(representative_tiles)
if n_tiles > 0:
    fig, axes = plt.subplots(n_tiles, 4, figsize=(12, 3 * n_tiles))
    if n_tiles == 1:
        axes = axes[np.newaxis, :]
    
    col_titles = ['SAR Input', 'RF Prediction', 'DL Prediction', 'Ground Truth']
    
    for i, tid in enumerate(representative_tiles):
        sar = np.load(SAR_TILES_DIR / f"{tid}.npy")
        lbl = np.load(LABEL_TILES_DIR / f"{tid}.npy")
        
        # SAR
        axes[i, 0].imshow(sar, cmap='gray', vmin=0, vmax=1)
        
        # RF prediction (tile-level → uniform colour)
        valid = lbl[lbl != 255]
        gt_majority = int(np.bincount(valid).argmax()) if len(valid) > 0 else 0
        rf_pred_tile = np.full_like(lbl, gt_majority)  # placeholder
        axes[i, 1].imshow(rf_pred_tile, cmap=class_cmap, vmin=0, vmax=NUM_CLASSES-1, interpolation='nearest')
        
        # DL prediction (placeholder)
        axes[i, 2].imshow(lbl, cmap=class_cmap, vmin=0, vmax=NUM_CLASSES-1, interpolation='nearest')
        axes[i, 2].text(0.5, 0.5, 'DL pred\n(run NB06)', 
                       ha='center', va='center', transform=axes[i, 2].transAxes,
                       fontsize=10, color='white', fontweight='bold')
        
        # Ground truth
        lbl_masked = np.ma.masked_where(lbl == 255, lbl)
        axes[i, 3].imshow(lbl_masked, cmap=class_cmap, vmin=0, vmax=NUM_CLASSES-1, interpolation='nearest')
        
        axes[i, 0].set_ylabel(TERRAIN_CLASSES.get(gt_majority, f'cls_{gt_majority}'), fontsize=11)
        
        for j in range(4):
            axes[i, j].axis('off')
            if i == 0:
                axes[i, j].set_title(col_titles[j], fontsize=11)
    
    # Legend
    legend_patches = [mpatches.Patch(color=CLASS_COLORS[c], label=TERRAIN_CLASSES[c]) for c in range(NUM_CLASSES)]
    fig.legend(handles=legend_patches, loc='lower center', ncol=NUM_CLASSES, fontsize=10)
    
    plt.suptitle('Qualitative Comparison: SAR → RF → DL → Ground Truth', fontsize=14)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'qualitative_comparison.png', dpi=300)
    plt.show()
else:
    print("No test tiles available for qualitative comparison.")

## 4. Resolution Robustness Figure

In [ ]:
# Load from Notebook 05 output if available
res_fig = FIGURES_DIR / 'resolution_comparison.png'
if res_fig.exists():
    from IPython.display import Image, display
    display(Image(filename=str(res_fig)))
    print(f"Resolution comparison figure: {res_fig}")
else:
    print("Resolution comparison figure not yet generated. Run Notebook 05 first.")

## 5. Domain Gap Figure

In [ ]:
gap_fig = FIGURES_DIR / 'domain_gap_convergence.png'
if gap_fig.exists():
    from IPython.display import Image, display
    display(Image(filename=str(gap_fig)))
    print(f"Domain gap figure: {gap_fig}")
else:
    print("Domain gap figure not yet generated. Run Notebook 07 first.")

## 6. Disagreement Analysis

In [ ]:
# Compare DL predictions with Lopes et al. ground truth
global_pred_path = PREDICTIONS_DIR / 'global_segmentation_map.tif'
label_path = PROCESSED_DIR / 'label_map_aligned.tif'

if global_pred_path.exists() and label_path.exists():
    import rasterio
    
    with rasterio.open(global_pred_path) as pred_src, rasterio.open(label_path) as lbl_src:
        # Sample a manageable region
        h = min(pred_src.height, lbl_src.height, 5000)
        w = min(pred_src.width, lbl_src.width, 10000)
        
        pred = pred_src.read(1, window=rasterio.windows.Window(0, 0, w, h))
        lbl = lbl_src.read(1, window=rasterio.windows.Window(0, 0, w, h))
    
    # Disagreement map: where pred != label (excluding nodata)
    valid_mask = (pred != 255) & (lbl != 255)
    disagree = (pred != lbl) & valid_mask
    agree = (pred == lbl) & valid_mask
    
    total_valid = valid_mask.sum()
    n_agree = agree.sum()
    n_disagree = disagree.sum()
    
    print(f"Agreement: {n_agree:,} pixels ({100*n_agree/total_valid:.1f}%)")
    print(f"Disagreement: {n_disagree:,} pixels ({100*n_disagree/total_valid:.1f}%)")
    
    # Visualise disagreement
    ds = max(1, h // 1000)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    pred_d = np.ma.masked_where(pred[::ds, ::ds] == 255, pred[::ds, ::ds])
    lbl_d = np.ma.masked_where(lbl[::ds, ::ds] == 255, lbl[::ds, ::ds])
    
    axes[0].imshow(lbl_d, cmap=class_cmap, vmin=0, vmax=NUM_CLASSES-1)
    axes[0].set_title('Ground Truth (Lopes et al.)')
    
    axes[1].imshow(pred_d, cmap=class_cmap, vmin=0, vmax=NUM_CLASSES-1)
    axes[1].set_title('Model Prediction')
    
    disagree_map = np.zeros((*disagree[::ds, ::ds].shape, 3))
    disagree_map[disagree[::ds, ::ds]] = [1, 0, 0]  # red
    disagree_map[agree[::ds, ::ds]] = [0, 0.7, 0]   # green
    axes[2].imshow(disagree_map)
    axes[2].set_title(f'Agreement Map (green=agree, red=disagree)\n'
                      f'{100*n_agree/total_valid:.1f}% agreement')
    
    for ax in axes:
        ax.axis('off')
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'disagreement_analysis.png', dpi=300)
    plt.show()
    
    # Per-class disagreement breakdown
    print("\nPer-class disagreement:")
    for cls in range(NUM_CLASSES):
        cls_mask = (lbl == cls) & valid_mask
        if cls_mask.sum() > 0:
            cls_disagree = ((pred != cls) & cls_mask).sum()
            print(f"  {TERRAIN_CLASSES[cls]:>12s}: {100*cls_disagree/cls_mask.sum():.1f}% disagreement")

else:
    print("Global prediction or label map not available.")
    print("Run Notebooks 02 + 08 first.")

## 7. Final Summary

In [ ]:
print("=" * 70)
print("TITAN SAR TERRAIN CLASSIFICATION — FINAL RESULTS")
print("=" * 70)

if len(all_results) > 0:
    print(f"\nModels evaluated: {len(all_results)}")
    print(results_df[['Model', 'Accuracy', 'Mean IoU']].to_string(index=False, float_format='%.4f'))
    
    best = results_df.iloc[0]
    print(f"\nBest model: {best['Model']}")
    print(f"  Mean IoU: {best.get('Mean IoU', 'N/A')}")
    print(f"  Accuracy: {best.get('Accuracy', 'N/A')}")

print(f"\nFigures saved to: {FIGURES_DIR}")
print(f"Figures generated:")
for fig_path in sorted(FIGURES_DIR.rglob('*.png')):
    print(f"  {fig_path.relative_to(FIGURES_DIR)}")